In [ ]:
from datasets import load_dataset

imdb_dataset=load_dataset('stanfordnlp/imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
split=imdb_dataset['train'].train_test_split(train_size=0.8,seed=42)

imdb_train_set=split['train']
imdb_test_set=split['test']
imdb_valid_set=imdb_dataset['test']

In [ ]:
print(imdb_train_set[1])

{'text': "'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!", 'label': 1}


In [ ]:
print(imdb_train_set[1]['text'])

'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!


In [ ]:
print(imdb_train_set[1]['label'])

1


BPE

In [ ]:
import tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
bpe_model=tokenizers.models.BPE(unk_token='<unk>')
bpe_model

BPE(dropout=None, unk_token="<unk>", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[])

In [ ]:
bpe_tokenizer=tokenizers.Tokenizer(bpe_model)
bpe_tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[], normalizer=None, pre_tokenizer=None, post_processor=None, decoder=None, model=BPE(dropout=None, unk_token="<unk>", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={}, merges=[]))

In [ ]:
bpe_tokenizer.pre_tokenizer=tokenizers.pre_tokenizers.Whitespace()

In [ ]:
special_tokens=['<pad>','<unk>']

In [ ]:
bpe_trainer=tokenizers.trainers.BpeTrainer(vocab_size=1000,special_tokens=special_tokens)


In [ ]:
train_reviews=[review['text'].lower() for review in imdb_train_set]
train_reviews[0]

'stage adaptations often have a major fault. they often come out looking like a film camera was simply placed on the stage (such as "night mother"). sidney lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. still, it\'s nice to look at for what it is. the chemistry between michael caine and christopher reeve is quite brilliant. the dynamics of their relationship are surprising. caine is fantastic as always, and reeve gets one of his few chances to really act.<br /><br />i confess that i\'ve never seen ira levin\'s play, but i hear that jay presson allen\'s adaptation is faithful. the script is incredibly convoluted, and keeps you guessing. "deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10'

In [ ]:
bpe_tokenizer.train_from_iterator(train_reviews,bpe_trainer)

In [ ]:
some_review='what an awesome movie! 😀'

bpe_encoding=bpe_tokenizer.encode(some_review)
print(bpe_encoding.tokens)

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']


In [ ]:
print(bpe_encoding.ids)

[303, 139, 373, 149, 240, 211, 4, 1]


In [ ]:
print(bpe_encoding.type_ids)

[0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
print(bpe_encoding.offsets)

[(0, 4), (5, 7), (8, 10), (10, 12), (12, 15), (16, 21), (21, 22), (23, 24)]


In [ ]:
print(bpe_encoding.attention_mask)

[1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
print(bpe_encoding.special_tokens_mask)

[0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
print(bpe_tokenizer.decode(bpe_encoding.ids))

what an aw es ome movie !


In [ ]:
bpe_encodings=bpe_tokenizer.encode_batch(train_reviews[:3])

In [ ]:
bpe_encodings[0].tokens

['st',
 'age',
 'ad',
 'ap',
 't',
 'ations',
 'of',
 'ten',
 'have',
 'a',
 'ma',
 'j',
 'or',
 'fa',
 'ult',
 '.',
 'they',
 'of',
 'ten',
 'come',
 'out',
 'looking',
 'like',
 'a',
 'film',
 'c',
 'amer',
 'a',
 'was',
 'sim',
 'p',
 'ly',
 'pl',
 'ac',
 'ed',
 'on',
 'the',
 'st',
 'age',
 '(',
 'such',
 'as',
 '"',
 'night',
 'mo',
 'ther',
 '"',
 ').',
 'sid',
 'ney',
 'lu',
 'me',
 't',
 "'",
 's',
 'dire',
 'ction',
 'keep',
 's',
 'the',
 'film',
 'al',
 'ive',
 ',',
 'which',
 'is',
 'especially',
 'dif',
 'f',
 'icul',
 't',
 'since',
 'the',
 'pict',
 'ure',
 'off',
 'er',
 'ed',
 'him',
 'no',
 'real',
 'ch',
 'all',
 'en',
 'ge',
 '.',
 'still',
 ',',
 'it',
 "'",
 's',
 'n',
 'ice',
 'to',
 'look',
 'at',
 'for',
 'what',
 'it',
 'is',
 '.',
 'the',
 'che',
 'mis',
 'try',
 'between',
 'm',
 'ich',
 'a',
 'el',
 'c',
 'ain',
 'e',
 'and',
 'chr',
 'is',
 'top',
 'her',
 'ree',
 've',
 'is',
 'quite',
 'br',
 'illi',
 'ant',
 '.',
 'the',
 'd',
 'y',
 'n',
 'am',
 'ics',

In [ ]:
bpe_tokenizer.enable_padding(
    pad_id=0,
    pad_token='<pad>'
)
bpe_tokenizer.enable_truncation(max_length=500)

In [ ]:
import torch
bpe_encodings=bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_batch_ids=torch.tensor([encoding.ids for encoding in bpe_encodings])
print(bpe_batch_ids)

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [ ]:
attention_mask=torch.tensor([encoding.attention_mask for encoding in bpe_encodings])
print(attention_mask)

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
lengths=attention_mask.sum(dim=-1)
print(lengths)

tensor([281, 114, 285])


BBPE

In [ ]:
bpe_tokenizer.pre_tokenizer=tokenizers.pre_tokenizers.ByteLevel()

In [ ]:
bpe_tokenizer.decoder=tokenizers.decoders.ByteLevel()

In [ ]:
special_tokens=['<pad>','<unk>']

bpe_trainer=tokenizers.trainers.BpeTrainer(vocab_size=1000,special_tokens=special_tokens)

In [ ]:
bpe_tokenizer.train_from_iterator(train_reviews,bpe_trainer)

In [ ]:
some_review='what an awesome movie!'

bpe_encoding=bpe_tokenizer.encode(some_review)
print(bpe_encoding.tokens)
print(bpe_encoding.ids)
print(bpe_tokenizer.decode(bpe_encoding.ids))

['Ġwhat', 'Ġan', 'Ġaw', 'es', 'ome', 'Ġmovie', '#']
[354, 216, 561, 148, 244, 232, 4]
 what an awesome movie#


WordPiece

In [ ]:
wordpiece_model=tokenizers.models.WordPiece(unk_token='[UNK]')
wordpiece_tokenizer=tokenizers.Tokenizer(wordpiece_model)

In [ ]:
wordpiece_tokenizer.pre_tokenizer=tokenizers.pre_tokenizers.Whitespace()

In [ ]:
wordpiece_trainer=tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"]
)

In [ ]:
wordpiece_tokenizer.train_from_iterator(train_reviews,wordpiece_trainer)

In [ ]:
some_review='what an awesome movie! 😀'

wordpiece_encoding=wordpiece_tokenizer.encode(some_review)
print(wordpiece_encoding.tokens)
print(wordpiece_encoding.ids)
print(wordpiece_tokenizer.decode(wordpiece_encoding.ids))

['what', 'an', 'aw', '##es', '##ome', 'movie', '!', '[UNK]']
[443, 312, 635, 257, 354, 331, 4, 1]
what an aw ##es ##ome movie !


In [ ]:
wordpiece_tokenizer.decoder=tokenizers.decoders.WordPiece(
    prefix="##"
)

wordpiece_trainer=tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"],
    continuing_subword_prefix="##"
)

wordpiece_tokenizer.train_from_iterator(train_reviews,wordpiece_trainer)

some_review='what an awesome movie! 😀'
encoding=wordpiece_tokenizer.encode(some_review)
print(encoding.tokens)
print(encoding.ids)
print(wordpiece_tokenizer.decode(encoding.ids))

['what', 'an', 'aw', '##es', '##ome', 'movie', '!', '[UNK]']
[443, 312, 635, 257, 354, 331, 4, 1]
what an awesome movie!


Unigram LLM

In [ ]:
unigram_model=tokenizers.models.Unigram()
unigram_tokenizer=tokenizers.Tokenizer(unigram_model)

In [ ]:
unigram_tokenizer.pre_tokenizer=tokenizers.pre_tokenizers.Metaspace(
    replacement="_",
    prepend_scheme='always'
)

In [ ]:
unigram_tokenizer.decoder=tokenizers.decoders.Metaspace(
    replacement="_",
    prepend_scheme='always'
)

In [ ]:
unigram_trainer=tokenizers.trainers.UnigramTrainer(
    vocab_size=1000,
    special_tokens=["<pad>", "<unk>"],
    unk_token="<unk>"
)

In [ ]:
unigram_tokenizer.train_from_iterator(train_reviews,unigram_trainer)

In [ ]:
some_review="what an awesome movie!😀"

enocding=unigram_tokenizer.encode(some_review)
print(enocding.tokens)
print(enocding.ids)
print(unigram_tokenizer.decode(enocding.ids,skip_special_tokens=True))

['_what', '_an', '_a', 'w', 'e', 'some', '_movie', '!', '😀']
[129, 76, 7, 40, 11, 711, 52, 103, 1]
what an awesome movie!


#Reusing Pretrained Tokenizers

In [ ]:
import transformers

In [ ]:
gpt2_tokenizer=transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_tokenizer

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [ ]:
gpt2_encoding = gpt2_tokenizer(
    train_reviews[:3],
    truncation=True,
    max_length = 500)

In [ ]:
gpt2_encoding

{'input_ids': [[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 10508, 355, 366, 3847, 2802, 11074, 9785, 1681, 46390, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 991, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 262, 16585, 1022, 285, 40302, 269, 5718, 290, 33826, 8803, 302, 44655, 318, 2407, 10457, 13, 262, 17262, 286, 511, 2776, 389, 6452, 13, 269, 5718, 318, 9623, 355, 1464, 11, 290, 302, 44655, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 72, 22127, 326, 1312, 1053, 1239, 1775, 4173, 64, 443, 7114, 338, 711, 11, 475, 1312, 3285, 326, 474, 323, 1803, 261, 477, 268, 338, 16711, 318, 17074, 13, 262, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 22595, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159, 29847, 1

In [ ]:
train_reviews[0]

'stage adaptations often have a major fault. they often come out looking like a film camera was simply placed on the stage (such as "night mother"). sidney lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. still, it\'s nice to look at for what it is. the chemistry between michael caine and christopher reeve is quite brilliant. the dynamics of their relationship are surprising. caine is fantastic as always, and reeve gets one of his few chances to really act.<br /><br />i confess that i\'ve never seen ira levin\'s play, but i hear that jay presson allen\'s adaptation is faithful. the script is incredibly convoluted, and keeps you guessing. "deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10'

In [ ]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
print(gpt2_token_ids)

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]


In [ ]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_tokenizer # wordpiece

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [ ]:
bert_encoding=bert_tokenizer(train_reviews[:3],
                             padding=True,
                             truncation=True,
                             max_length=500,
                             return_tensors="pt",
                            #  add_special_tokens=False
)

#101-CLS
#102-SEP

In [ ]:
bert_encoding

{'input_ids': tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  

In [ ]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")
albert_tokenizer # unigram

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

AlbertTokenizer(name_or_path='albert-base-v2', vocab_size=30000, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '<unk>', 'sep_token': '[SEP]', 'pad_token': '<pad>', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [ ]:
albert_encoding = albert_tokenizer(train_reviews[:3],
                                  padding=True,
                                  truncation=True,
                                  max_length=500)

In [ ]:
albert_encoding

{'input_ids': [[2, 876, 5004, 18, 478, 57, 21, 394, 4173, 9, 59, 478, 340, 70, 699, 101, 21, 171, 3336, 23, 1659, 1037, 27, 14, 876, 13, 5, 4289, 28, 13, 7, 4893, 449, 7, 6, 9, 12508, 1612, 5909, 22, 18, 1400, 8968, 14, 171, 2481, 15, 56, 25, 1118, 1956, 179, 14, 2151, 1434, 61, 90, 683, 2404, 9, 174, 15, 32, 22, 18, 2210, 20, 361, 35, 26, 98, 32, 25, 9, 14, 5427, 128, 832, 22427, 17, 4479, 24604, 25, 1450, 7472, 9, 14, 12289, 16, 66, 1429, 50, 12891, 9, 22427, 25, 10356, 28, 550, 15, 17, 24604, 3049, 53, 16, 33, 310, 11285, 20, 510, 601, 9, 1, 5145, 13, 118, 1, 5145, 13, 118, 1, 49, 14586, 30, 31, 22, 195, 243, 541, 16216, 18863, 22, 18, 418, 15, 47, 31, 990, 30, 3361, 901, 218, 3675, 22, 18, 5004, 25, 10763, 9, 14, 3884, 25, 13003, 1065, 16261, 1427, 15, 17, 8968, 42, 19523, 9, 13, 7, 13921, 16514, 7, 25, 40, 7135, 102, 15677, 171, 15, 17, 25, 5773, 26, 1212, 65, 3047, 16, 876, 17, 2324, 9, 1, 5145, 13, 118, 1, 5145, 13, 118, 1, 465, 9, 300, 70, 16, 332, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
hf_tokenizer=transformers.PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer
)

In [ ]:
hf_encoding=hf_tokenizer(
    train_reviews[:3],
    padding=True,
    truncation=True,
    max_length=500
)

In [ ]:
hf_encoding

{'input_ids': [[196, 499, 460, 40, 351, 862, 159, 59, 146, 264, 129, 282, 49, 149, 462, 611, 15, 297, 159, 59, 146, 885, 317, 466, 147, 314, 129, 222, 842, 132, 40, 206, 662, 55, 186, 288, 641, 160, 190, 134, 196, 499, 247, 58, 60, 199, 213, 250, 53, 327, 140, 516, 3, 581, 133, 217, 553, 64, 162, 321, 235, 200, 501, 201, 637, 559, 58, 134, 222, 272, 347, 13, 412, 167, 815, 259, 980, 324, 384, 172, 611, 969, 134, 154, 521, 418, 506, 132, 160, 416, 352, 508, 236, 218, 146, 285, 15, 622, 13, 173, 200, 169, 470, 157, 466, 256, 214, 354, 173, 167, 15, 134, 236, 263, 312, 502, 849, 140, 333, 40, 306, 144, 302, 44, 155, 236, 57, 312, 310, 336, 189, 44, 187, 167, 803, 407, 243, 48, 294, 15, 134, 153, 64, 53, 227, 172, 58, 159, 408, 768, 320, 706, 598, 253, 701, 856, 150, 147, 15, 144, 302, 44, 167, 142, 294, 318, 172, 213, 869, 13, 155, 189, 44, 187, 889, 277, 159, 252, 750, 236, 158, 534, 157, 402, 278, 238, 171, 215, 171, 175, 48, 313, 45, 284, 191, 136, 656, 568, 566, 136, 258, 332, 61, 131

In [ ]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [review["label"] for review in batch]

    encodings = tokenizer(
        reviews,
        padding=True,
        truncation=True,
        max_length=200,
        return_tensors="pt"
    )

    labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return encodings, labels


In [ ]:
from torch.utils.data import DataLoader

batch_size=256
imdb_train_loader = DataLoader(
    imdb_train_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True
)

imdb_valid_loader = DataLoader(
    imdb_valid_set,
    batch_size=batch_size,
    collate_fn = collate_fn,
    shuffle=False

)

imdb_test_loader = DataLoader(
    imdb_test_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False
)

In [ ]:
import torch.nn as nn

In [ ]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=2, hidden_dim=2, pad_id=0, dropout=0.2):
        super().__init__()

        self.embed= nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id)

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout

        )
        self.output = nn.Linear(hidden_dim, 1)


    def forward( self, encodings):
        embeddings=self.embed(encodings["input_ids"])
        _outputs,hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])

In [ ]:
import torch
from torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence

In [ ]:
sequences = torch.tensor([
    [1, 2, 0, 0],
    [5, 6, 7, 8]
])

packed = pack_padded_sequence(
    sequences,
    lengths=(2, 4),
    enforce_sorted=False,
    batch_first=True
)

print(packed)

PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))


In [ ]:
padded, lengths = pad_packed_sequence(
    packed,
    batch_first=True
)

print(padded)
print(lengths)

tensor([[1, 2, 0, 0],
        [5, 6, 7, 8]])
tensor([2, 4])


In [ ]:
class SentimentAnalysisModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]

        embeddings = self.embed(input_ids)

        lengths = attention_mask.sum(dim=1)

        packed_embeddings = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed_embeddings)

        last_hidden_state = hidden_states[-1]

        logits = self.output(last_hidden_state)

        return logits

In [ ]:
vocab_size = bert_tokenizer.vocab_size
pad_id = bert_tokenizer.pad_token_id

model = SentimentAnalysisModel(
    vocab_size=vocab_size,
    n_layers=2,
    embed_dim=128,
    hidden_dim=64,
    pad_id=pad_id,
    dropout=0.2
)


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=model.to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model,dataloader,optimizer,criterion,device):
  model.train()

  total_loss=0
  correct=0
  total=0

  for encodings,labels in dataloader:
    encodings={key:value.to(device) for key,value in encodings.items()}
    labels=labels.to(device)

    logits=model(encodings)
    loss=criterion(logits,labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss+=loss.item()

    preds=torch.round(torch.sigmoid(logits))
    correct+=(preds==labels).sum().item()
    total+=labels.size(0)
  avg_loss=total_loss/len(dataloader)
  accuracy=correct/total
  return avg_loss,accuracy


In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for encodings, labels in dataloader:
            encodings = {
                key: value.to(device)
                for key, value in encodings.items()
            }
            labels = labels.to(device)

            logits = model(encodings)

            loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        imdb_train_loader,
        optimizer,
        criterion,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        imdb_valid_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )

Epoch 1/5 | Train Loss: 0.6807 | Train Acc: 0.5544 | Valid Loss: 0.6695 | Valid Acc: 0.6030
Epoch 2/5 | Train Loss: 0.6133 | Train Acc: 0.6700 | Valid Loss: 0.6105 | Valid Acc: 0.6730
Epoch 3/5 | Train Loss: 0.5238 | Train Acc: 0.7518 | Valid Loss: 0.5244 | Valid Acc: 0.7554
Epoch 4/5 | Train Loss: 0.4406 | Train Acc: 0.8065 | Valid Loss: 0.4776 | Valid Acc: 0.7811
Epoch 5/5 | Train Loss: 0.3631 | Train Acc: 0.8518 | Valid Loss: 0.4594 | Valid Acc: 0.7958


In [ ]:
def predict_sentiment(model, tokenizer, sentences, device, max_length=200):
    model.eval()

    encodings = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    encodings = {
        key: value.to(device)
        for key, value in encodings.items()
    }

    with torch.no_grad():
        logits = model(encodings)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long().squeeze(1)

    label_names = {
        0: "Negative",
        1: "Positive"
    }

    results = []

    for sentence, prob, pred in zip(sentences, probs.squeeze(1), preds):
        results.append({
            "sentence": sentence,
            "probability_positive": prob.item(),
            "predicted_label_id": pred.item(),
            "predicted_label": label_names[pred.item()]
        })

    return results

In [ ]:
test_sentences = [
    "This movie was amazing and I really enjoyed it.",
    "The film was boring and too long.",
    "I loved the acting but the story was weak.",
    "Absolutely terrible movie. I would not recommend it.",
    "One of the best films I have watched recently."
]

results = predict_sentiment(
    model=model,
    tokenizer=bert_tokenizer,
    sentences=test_sentences,
    device=device
)

for r in results:
    print("Sentence:", r["sentence"])
    print("Positive probability:", round(r["probability_positive"], 4))
    print("Predicted label id:", r["predicted_label_id"])
    print("Predicted label:", r["predicted_label"])
    print("-" * 50)

Sentence: This movie was amazing and I really enjoyed it.
Positive probability: 0.8776
Predicted label id: 1
Predicted label: Positive
--------------------------------------------------
Sentence: The film was boring and too long.
Positive probability: 0.127
Predicted label id: 0
Predicted label: Negative
--------------------------------------------------
Sentence: I loved the acting but the story was weak.
Positive probability: 0.5836
Predicted label id: 1
Predicted label: Positive
--------------------------------------------------
Sentence: Absolutely terrible movie. I would not recommend it.
Positive probability: 0.1252
Predicted label id: 0
Predicted label: Negative
--------------------------------------------------
Sentence: One of the best films I have watched recently.
Positive probability: 0.8584
Predicted label id: 1
Predicted label: Positive
--------------------------------------------------


In [ ]:
class SentimentAnalysisModel(nn.Module):
  def __init__(self,vocab_size,n_layers=2,embed_dim=128,hidden_dim=64,pad_id=0,dropout=0.2):
    super().__init__()
    self.embed=nn.Embedding(num_embeddings=vocab_size,embedding_dim=embed_dim,padding_idx=pad_id)
    self.gru=nn.GRU(input_size=embed_dim,hidden_size=hidden_dim,num_layers=n_layers,batch_first=True,bidirectional=True,dropout=dropout)
    self.output=nn.Linear(2*hidden_dim,1)
  def forward(self,encodings):
    input_ids=encodings['input_ids']

    attention_mask=encodings['attention_mask']
    embeddings=self.embed(input_ids)
    lengths=attention_mask.sum(dim=1)

    packed_embeddings=pack_padded_sequence(embeddings,lengths=lengths.cpu(),batch_first=True,enforce_sorted=False)
    _outputs,hidden_states=self.gru(packed_embeddings)
    n_dims=self.output.in_features
    top_states=hidden_states[-2:].permute(1,0,2).reshape(-1,n_dims)
    logits=self.output(top_states)
    return logits

##Reusing Pretrained Embeddings and Language models

In [ ]:
bert_model=transformers.AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
bert_model.embeddings.word_embeddings

Embedding(30522, 768, padding_idx=0)

In [ ]:
bert_model.embeddings.word_embeddings.weight.data

tensor([[-0.0102, -0.0615, -0.0265,  ..., -0.0199, -0.0372, -0.0098],
        [-0.0117, -0.0600, -0.0323,  ..., -0.0168, -0.0401, -0.0107],
        [-0.0198, -0.0627, -0.0326,  ..., -0.0165, -0.0420, -0.0032],
        ...,
        [-0.0218, -0.0556, -0.0135,  ..., -0.0043, -0.0151, -0.0249],
        [-0.0462, -0.0565, -0.0019,  ...,  0.0157, -0.0139, -0.0095],
        [ 0.0015, -0.0821, -0.0160,  ..., -0.0081, -0.0475,  0.0753]])

In [ ]:
nn.Embedding.from_pretrained(
    bert_model.embeddings.word_embeddings.weight.data,
    freeze=True
)

Embedding(30522, 768)

In [ ]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, dropout=0.2 ):
        super().__init__()

        weights = pretrained_embeddings.weight.data

        self.embed = nn.Embedding.from_pretrained(
            weights,
            freeze=True
        )

        embed_dim = weights.shape[-1]

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])

        lengths = encodings["attention_mask"].sum(dim=1)

        packed = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed)

        return self.output(hidden_states[-1])

In [ ]:
imdb_model_bert_embeds = SentimentAnalysisModelPreEmbeds(
    bert_model.embeddings.word_embeddings
).to(device)# traine yolla

In [ ]:
bert_encoding = bert_tokenizer(
    train_reviews[:3],
    padding=True,
    max_length=200,
    truncation=True,
    return_tensors="pt"
)
bert_model=transformers.AutoModel.from_pretrained("bert-base-uncased")

bert_output = bert_model(**bert_encoding)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
transformers.AutoModel.from_pretrained("bert-base-uncased").config.hidden_size

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


768

In [ ]:
class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()

        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")

        embed_dim = self.bert.config.hidden_size

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        contextualized_embeddings = self.bert(**encodings).last_hidden_state

        lengths = encodings["attention_mask"].sum(dim=1)

        packed = pack_padded_sequence(
            contextualized_embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed) # output each step hidden state [1,2,3,x]

        return self.output(hidden_states[-1])

In [ ]:
#cls token

In [ ]:
class SentimentAnalysisBertCLS(nn.Module):
  def __init__(self):
    super().__init__()
    self.bert=transformers.AutoModel.from_pretrained('bert-base-uncased')
    self.output=nn.Linear(in_features=self.bert.config.hidden_size,out_features=1)
  def forward(self,encodings):
    bert_output=self.bert(**encodings)
    return self.output(bert_output.last_hidden_state[:,0,:])

In [ ]:
#pooler layer

In [ ]:
class SentimentAnalysisModelModelBertCLS(nn.Module):
  def __init__(self):
    super().__init__()
    self.bert=transformers.AutoModel.from_pretrained('bert-base-uncased')
    self.output=nn.Linear(in_features=self.bert.config.hidden_size,out_features=1)

  def forward(self,encodings):
    bert_output=self.bert(**encodings)
    cls_vector=bert_output.pooler_output
    return self.output(bert_output.pooler_output)

## Task Specific

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

bert_tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
bert_for_binary_clf=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2,dtype=torch.float16 if device=='cuda' else torch.float32).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
review = ["This was not a great movie!"]


encoding = bert_tokenizer(
    review,
    padding=True,
    truncation=True,
    max_length=200,
    return_tensors="pt"
)

In [ ]:
encoding = {key: value.to(device) for key, value in encoding.items()}

with torch.no_grad():
    output = bert_for_binary_clf(**encoding)

logits = output.logits
probs = torch.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1).item()

In [ ]:
label_names = ["negative", "positive"]

In [ ]:
logits

tensor([[ 0.7263, -0.4250]], device='cuda:0')

In [ ]:
probs

tensor([[0.7597, 0.2403]], device='cuda:0')

In [ ]:
label_names[predicted_class]

'negative'

##The Trainer Api

In [ ]:
import numpy as np
import torch
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
device='cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
bert_tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
bert_for_binary_clf=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=2,dtype=torch.float16 if device=='cuda' else torch.float32).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_batch(batch):
    return bert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=200
    )

In [ ]:
tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched=True)
tok_imdb_test_set = imdb_test_set.map(tokenize_batch, batched=True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
data_collator=DataCollatorWithPadding(tokenizer=bert_tokenizer)

In [ ]:
def compute_accuracy(pred):
    logits = pred.predictions
    labels = pred.label_ids

    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()

    return {"accuracy": accuracy}

In [ ]:
!pip install wandb

In [ ]:
import wandb
wandb.login()
#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

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nihatqasimov (nihatqasimov-unec-business-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
train_args = TrainingArguments(
    output_dir="my_imdb_model", # it holds checkpoints and logs...
    num_train_epochs=2,  # pass 2 times for whole dataset
    per_device_train_batch_size=16, # for each step model takes 16 reviews
    per_device_eval_batch_size=16,
    eval_strategy="epoch", # her epoch sonuna validation acc yazilir
    logging_strategy="epoch",
    save_strategy="epoch",  #checkpoint
    load_best_model_at_end=True, # not last one but the best one
    metric_for_best_model="accuracy",
    report_to="wandb",
    run_name="bert-imdb-run-1"
)

In [ ]:
data_collator=DataCollatorWithPadding(tokenizer=bert_tokenizer)

In [ ]:
trainer = Trainer(
    model=bert_for_binary_clf,
    args=train_args,
    train_dataset=tok_imdb_train_set,
    eval_dataset=tok_imdb_valid_set,
    compute_metrics=compute_accuracy,
    data_collator=data_collator
)

In [ ]:
train_output=trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.774878,nan,0.500000
2,0.000000,nan,0.500000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

In [ ]:
trainer.evaluate(tok_imdb_valid_set)

Training Loss,Validation Loss,Epoch,Accuracy
0.000000,nan,2,0.500000


{'eval_loss': nan, 'eval_accuracy': 0.5}

In [ ]:
review = ["This was not a great movie!"]

inputs = bert_tokenizer(
    review,
    return_tensors="pt",
    truncation=True,
    max_length=200,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

bert_for_binary_clf.eval()

with torch.no_grad():
    outputs = bert_for_binary_clf(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)
pred = torch.argmax(probs, dim=-1).item()

label_names = ["negative", "positive"]

print("Probabilities:", probs)
print("Prediction:", label_names[pred])

Probabilities: tensor([[nan, nan]], device='cuda:0', dtype=torch.float16)
Prediction: negative
